# Outside Bar Continuation on SPY
## Strategy Brief
The Outside Bar Continuation strategy identifies potential continuation moves following an outside bar pattern on the SPY ETF. An outside bar occurs when the high is higher and the low is lower than the previous bar, indicating increased volatility and potential trend continuation. The strategy enters a long position if the price moves above the high of the outside bar and exits if it moves below the low. Historical testing suggests this pattern may capture short-term momentum in the SPY.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters and constants used throughout the strategy implementation. This includes the lookback period for identifying outside bars and the initial capital for backtesting.

In [ ]:
# Configuration
LOOKBACK_PERIOD = 1  # Lookback period for identifying outside bars
INITIAL_CAPITAL = 10000  # Initial capital for backtesting
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'

## PHASE 2 - Data Exploration
We will download historical data for SPY using yfinance, calculate the outside bar pattern, and visualize it overlaid on the price chart.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Calculate Outside Bar
outside_bar = (data['High'] > data['High'].shift(1)) & (data['Low'] < data['Low'].shift(1))

data['OutsideBar'] = outside_bar

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.scatter(data.index, data['Close'][outside_bar], color='red', label='Outside Bar', marker='o')
plt.title('SPY Price with Outside Bars')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We define the logic for entering and exiting trades based on the outside bar pattern. The signal series indicates potential entry points, and the positions series is used to track open positions.

In [ ]:
# Signal Series
signal = pd.Series(index=data.index, data=0)

# Entry logic: Buy if price goes above the high of the outside bar
signal[data['Close'] > data['High'].shift(1)] = 1

# Exit logic: Exit if price goes below the low of the outside bar
signal[data['Close'] < data['Low'].shift(1)] = 0

# Positions Series
positions = signal.shift(1).fillna(0)

## PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns based on the positions series and plot the resulting equity curve.

In [ ]:
# Calculate daily returns
returns = data['Close'].pct_change()

# Strategy returns
strategy_returns = positions * returns

# Equity curve
equity_curve = (1 + strategy_returns).cumprod() * INITIAL_CAPITAL

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Equity Curve')
plt.title('Strategy Equity Curve')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance using key metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown, and compare it with a buy-and-hold strategy.

In [ ]:
import numpy as np

# Calculate performance metrics
cagr = (equity_curve[-1] / INITIAL_CAPITAL) ** (1 / ((equity_curve.index[-1] - equity_curve.index[0]).days / 365.25)) - 1
sharpe_ratio = np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(252)
sortino_ratio = np.mean(strategy_returns) / np.std(strategy_returns[strategy_returns < 0]) * np.sqrt(252)
max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
calmar_ratio = cagr / abs(max_drawdown)

# Buy and hold strategy
buy_and_hold_returns = data['Close'].pct_change().fillna(0)
buy_and_hold_equity = (1 + buy_and_hold_returns).cumprod() * INITIAL_CAPITAL
buy_and_hold_cagr = (buy_and_hold_equity[-1] / INITIAL_CAPITAL) ** (1 / ((buy_and_hold_equity.index[-1] - buy_and_hold_equity.index[0]).days / 365.25)) - 1

# Comparison table
comparison = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': [cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown],
    'Buy and Hold': [buy_and_hold_cagr, np.nan, np.nan, np.nan, (buy_and_hold_equity / buy_and_hold_equity.cummax() - 1).min()]
})

print(comparison)

## PHASE 6 - Deploy & Monitor
We create a function to download the last 60 days of SPY data, compute today's signal, and print the current position based on the strategy logic.

In [ ]:
def get_today_signal():
    # Download last 60 days of data
    recent_data = yf.download('SPY', period='60d')
    
    # Calculate Outside Bar
    recent_outside_bar = (recent_data['High'] > recent_data['High'].shift(1)) & (recent_data['Low'] < recent_data['Low'].shift(1))
    
    # Determine today's signal
    if recent_data['Close'][-1] > recent_data['High'].shift(1)[-1]:
        print('Signal: Buy')
    elif recent_data['Close'][-1] < recent_data['Low'].shift(1)[-1]:
        print('Signal: Sell')
    else:
        print('Signal: Hold')

get_today_signal()